<a href="https://colab.research.google.com/github/NanoBurgos/Penguin_Academy/blob/main/challenges/desafio5_conexion0/Python_Socket_TCP_Chat_en_consola_(CMD).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.youtube.com/watch?v=FTdii0o5vBM

# **Python Socket - TCP Chat en consola (CMD)**

In [ ]:
#archivo del server server.py



import socket
import threading

host = '127.0.0.1'
port = 55555

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

server.bind((host, port))
server.listen()
print(f"Server running on {host}:{port}")

clients = []
usernames = []

def broadcast(message, _client):
  for client in clients:
    if client != _client:
      client.send(message)

def handle_messages(client):
  while True:
    try:
      message = client.recv(1024)
      broadcast(message, client)
    except:
      index = clients.index(client)
      username = usernames[index]
      broadcast(f"Chatbot: {username} disconected".encode('utf-8'), client)
      clients.remove(client)
      usernames.remove(username)
      client.close()
      break

def receive_connections():
  while True:
    client, address = server.accept()

    client.send("@username".encode("utf-8"))
    username = client.recv(1024).decode('utf-8')

    clients.append(client)
    usernames.append(username)

    print(f"{username} is connected with {str(address)}")

    message = f"ChatBot: {username} joined the chat!".encode("utf-8")
    broadcast(message, client)
    client.send("Connected to server".encode("utf-8"))

    thread = threading.Thread(target=handle_messages, args=(client,))
    thread.start()

receive_connections()












In [ ]:
#archivo del cliente client.py

import socket
import threading

username = input("Enter your name: ")

host = '127.0.0.1'
port = 55555

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((host, port))

def receive_messages():
  while True:
    try:
      message = client.recv(1024).decode('utf-8')

      if message == "@username":
        client.send(username.encode("utf-8"))

      else:
        print(message)

    except:
        print("An error Ocurred")
        client.close()
        break

def write_messages():
  while True:
    message = f"{username}: {input('')}"
    client.send(message.encode("utf-8"))

receive_thread = threading.Thread(target=receive_messages)
receive_thread.start()

write_thread = threading.Thread(target=write_messages)
write_thread.start()
